In [ ]:
import os
import csv
import openai
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL", ".....")

client = openai.OpenAI(api_key=api_key, base_url=base_url)

MODEL_NAME = "gpt-4o-mini"

LINES_PER_CALL = 15  
TOTAL_LINES = 3000    
OUTPUT_FILE = "Generated_Dataset_a_3000.csv"

SYSTEM_PROMPT = "You are a high-quality dataset generator."
USER_PROMPT_TEMPLATE = """
Task

Generate a high-quality parallel corpus for Romanized Nepali → Devanagari transliteration, intended for machine-learning and NLP research.
The dataset must preserve semantic correctness, grammatical validity, and phonetic fidelity.

Each output line must contain one meaningful sentence pair in the exact format:

romanized_nepali_sentence, devanagari_sentence


No extra text. No numbering. No explanations.

Core Objective

The goal is to train transliteration and normalization models, not language models.
Therefore:

Semantic correctness is mandatory

Grammatical correctness is mandatory

Phonetic → Maatraa mapping must be exact

Garbage data is strictly forbidden

If a sentence sounds unnatural or would never be spoken by a Nepali speaker, do not generate it.

Language Constraints
1. Language Composition

Sentences must be pure Nepali

DO NOT use English words unless explicitly instructed

Use natural spoken Nepali sentence structure

2. Romanization Rules (STRICT)

Follow phonetic Romanized Nepali, respecting vowel length and consonant distinction.

Mandatory rules:

aa → आ

a → अ

i / ee → इ / ई

u / uu → उ / ऊ

e → ए

ai → ऐ

o → ओ

au → औ

Consonants:

chh → छ

ch → च

kshy → क्ष

Never use x

Respect retroflex vs dental sounds where possible

Transliterate the english words to their devanagari form, not their nepali meaning. 
For example:
class -> क्लास
bottle -> बोतल
education -> एजुकेसन

3. MAATRAA ENFORCEMENT (CRITICAL)

You must strictly follow the vowel–maatra mapping rules defined in the uploaded file:

MAATRAAS.json


This file is the single source of truth for:

vowel length

दीर्घ / ह्रस्व distinctions

 If a Romanized vowel is wrong, the Devanagari output must be the corrected version. (you should auto-correct).

4. Verb Usage Rules (VERY IMPORTANT)

You are NOT allowed to use bare infinitives unnaturally.

Instead:

Use real conjugated forms that appear in spoken Nepali

Examples (illustrative, not exhaustive):

sutnu → sute, sutera, sutaayeँ, sutaaudaa

uthnu → uthe, uthera, uthdai

janu → gaye, gayera, jaandai

khanu → khaye, khayau, khada

padhnu → padhe, padhdai, padhyau

thichnu → thichyau

ladnu → ladyo

sikaunu → sikaaudai

banaunu → banaaudai, banaaunchha

 Never generate constructions like:

“ma sutnu bhayo”

“ma janu gare”

Any honorific mismatch or tense error

5. Nasalization & Phonetic Ambiguity (ENCOURAGED)

Actively use words with:

nasal sounds

अनुस्वार / चन्द्रबिन्दु

long vowels + suffixes

Examples of valid lexical patterns:
aaunchha, aaundaina, suhaaunchha, bhitraaunchhu, ghataaunchha, haanschha, heraunlaa, dekhaaunchhu, pattyaaundaina, banaaundai chha, lyaaundaina, jaandai chhu, haansdinu, guraans, heraunlaa, gaaunmaa, paaunchhu, jitaaunchha, hasaaunthe

They correspond to:
आउँछ, आउँदैन, सुहाउँछ, भित्राउँछु, घटाउँछ, हाँस्छ, हेरौँला, देखाउँछु, पत्याउँदैन, बनाउँदै छ, ल्याउँदैन, जाँदै छु, हाँस्दिनु, गुराँस, हेरौँला,गाउँमा, पाउँछु, जिताउँछ, हँसाउथे

Occasional spelling, maatraa's, and phonetic spelling variation are allowed, but:

The sentence's meaning must remain intact

6. Preserving the consistency rather than pure transliteration only

For example:

Even if Input is: 
uniharule khaana khaayo 

The output should be like: उनिहरुले खाना खाए 

Because "khaayo" or "खायो" goes with singular nouns like "रामले", "उस्ले", etc,. The accurate one is "khaaye" or "खाए" which goes with plural noun like "uniharu" or "उनिहरु". So, even if the input is inconsistent, correct the context in the output. Give context-aware outputs, not only transliteration of the romanized sentence.

7. Sentence Quality Requirements

Every sentence must:

Make real-world sense

Have clear subject–verb agreement

Contain occasional spelling variations and mistakes

Sound natural to a native Nepali speaker

Represent daily life, culture, education, work, society, emotions

Avoid:

robotic templates

repeated structures

meaningless filler

8. Output Format (STRICT)

One sentence per line

Around 20 sentences per response

The sentences should contain at least 20-25 words each, this is a strict requirement to maintain data quality and richness.
make sure there is no commas between the lines (This is strict and must be followed) and the sentences are separated by newlines only.
Exactly one comma separating the pair

No extra commas


No numbering

No markdown

Example (format only):

hamilai bholi bhetne chha karyalay ma ani tyaha kehi mahattwaapurna kura chhalachhal garaune chha,हामीलाई भोली भेट्ने छ कार्यालय म अनि त्यहाँ केही महत्त्वपूर्ण कुरा छलछल गराउने छ
usle akhir ma bolayo malai tara mero sanga baat garna man thiyena usle gareko galti pani,उसले अखिर म बोलायो मलाई तर मेरो संग बात गर्न मन थिएन उसले गरेको गल्ति पनि

9. Failure Condition

If you cannot:

guarantee semantic correctness

obey MAATRAAS.json

generate natural Nepali

Then do not generate output.

High-quality data > high quantity.

Generate exactly {n} sentence pairs with length at least 20-25 words each.
"""

def generate_batch(n_lines: int) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": USER_PROMPT_TEMPLATE.format(n=n_lines),
            },
        ],
        temperature=0.7,
    )

    return response.choices[0].message.content.strip()

import re

def parse_output(text: str):
    rows = []
    lines = text.split("\n")

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Find first Devanagari character
        match = re.search(r'[\u0900-\u097F]', line)
        if not match:
            continue

        idx = match.start()

        roman = line[:idx].strip(" ,")
        devanagari = line[idx:].strip(" ,")

        if roman and devanagari:
            rows.append([roman, devanagari])

    return rows

def generate_dataset():
    all_rows = []

    calls_needed = TOTAL_LINES // LINES_PER_CALL
    if TOTAL_LINES % LINES_PER_CALL != 0:
        calls_needed += 1

    for i in range(calls_needed):
        print(f"Generating batch {i+1}/{calls_needed}...")

        output = generate_batch(LINES_PER_CALL)
        rows = parse_output(output)

        all_rows.extend(rows)

        print(f"Collected {len(all_rows)} rows so far")


    all_rows = all_rows[:TOTAL_LINES]

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["romanized", "devanagari"]) 
        writer.writerows(all_rows)

    print(f"Saved {len(all_rows)} rows to {OUTPUT_FILE}")


if __name__ == "__main__":
    generate_dataset()

Generating batch 1/200...
Collected 15 rows so far
Generating batch 2/200...
Collected 30 rows so far
Generating batch 3/200...
Collected 45 rows so far
Generating batch 4/200...
Collected 60 rows so far
Generating batch 5/200...
Collected 75 rows so far
Generating batch 6/200...
Collected 90 rows so far
Generating batch 7/200...
Collected 105 rows so far
Generating batch 8/200...
Collected 119 rows so far
Generating batch 9/200...
Collected 134 rows so far
Generating batch 10/200...
Collected 149 rows so far
Generating batch 11/200...
Collected 164 rows so far
Generating batch 12/200...
Collected 179 rows so far
Generating batch 13/200...
Collected 194 rows so far
Generating batch 14/200...
Collected 209 rows so far
Generating batch 15/200...
Collected 224 rows so far
Generating batch 16/200...
Collected 238 rows so far
Generating batch 17/200...
Collected 253 rows so far
Generating batch 18/200...
Collected 268 rows so far
Generating batch 19/200...
Collected 282 rows so far
Generati

In [2]:
import pandas as pd
import os

# Folder containing CSV files
folder_path = "/home/sagargiri/Documents/Academic Projects/Transliteration/generated_dataset"

# List to store DataFrames
dataframes = []

# Loop through all files in the folder
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        dataframes.append(df)

# Combine all DataFrames
merged_df = pd.concat(dataframes, ignore_index=True)

# Save to a new CSV file
output_path = os.path.join(folder_path, "generated_only_merged.csv")
merged_df.to_csv(output_path, index=False)

print("All CSV files merged successfully!")

All CSV files merged successfully!
